[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-bayesian-gp.ipynb)

# The Bayesian Approach & Gaussian Processes

*AIBits Academy · Machine Learning End To End · ⚠ Advanced Topic*

Every model so far has returned a single "best" number for each parameter and each prediction. The Bayesian approach asks a different question: what's the full range of plausible answers, and how confident should we really be?

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

> **⚠ Why This Page Is Marked "Advanced"**
>
> Every model earlier in this course (Linear Regression, Random Forest, XGBoost, ...) is fit via a single optimization that returns *point estimates* — one number per parameter, one number per prediction. The Bayesian approach instead treats parameters as random variables with a full probability distribution, which requires a genuine shift in mental model, not just a new library call. This page introduces the idea at a foundational level.

## The Bayesian Idea — A Distribution, Not a Number

Bayes' rule turns a **prior** belief about parameters θ into a **posterior** belief, updated by observed data D:

$$P(\theta\mid D) = \dfrac{P(D\mid\theta)\cdot P(\theta)}{P(D)} \qquad \text{posterior} \propto \text{likelihood}\times\text{prior}$$

Every model up to this point implicitly picks a single θ that maximizes the likelihood (or a regularized version of it) and then discards all information about how confident that estimate is. The Bayesian approach keeps the entire posterior distribution — which means every prediction naturally comes with a genuine uncertainty estimate, not a number bolted on afterward.

## Bayesian Linear Regression — Watching the Posterior Narrow

Bayesian Ridge Regression puts a prior over the regression coefficients and returns both a posterior mean (comparable to the OLS/Ridge point estimate) *and* a posterior standard deviation for each coefficient — a direct measure of how uncertain that coefficient still is, given the data seen so far:

The lesson refers to two applicant tables (12 and 300 applicants) drawn from the same relationship. We generate them here.

In [ ]:
import numpy as np
rng = np.random.default_rng(5)

def make_applicants(n):
    income = rng.uniform(4, 30, n)                    # lakhs per year
    cibil = rng.uniform(600, 800, n) / 100            # score in hundreds, to keep the two scales comparable
    loan = 2.0 * income + 5.0 * cibil + rng.normal(0, 3, n)
    return np.column_stack([income, cibil]), loan

income_cibil_small, loan_amount_small = make_applicants(12)
income_cibil_large, loan_amount_large = make_applicants(300)
print(income_cibil_small.shape, income_cibil_large.shape)

In [ ]:
import numpy as np
from sklearn.linear_model import BayesianRidge

# HDFC Bank: loan_amount_lakhs predicted from applicant income_lakhs and cibil_score
# Small sample, n=12 applicants
X_small, y_small = income_cibil_small, loan_amount_small   # (12, 2) and (12,)
br_small = BayesianRidge().fit(X_small, y_small)
coef_std_small = np.sqrt(np.diag(br_small.sigma_))
print("n=12  coef mean:", np.round(br_small.coef_, 4))
print("n=12  coef std :", np.round(coef_std_small, 4))

# Same underlying relationship, n=300 applicants
X_large, y_large = income_cibil_large, loan_amount_large   # (300, 2) and (300,)
br_large = BayesianRidge().fit(X_large, y_large)
coef_std_large = np.sqrt(np.diag(br_large.sigma_))
print("n=300 coef mean:", np.round(br_large.coef_, 4))
print("n=300 coef std :", np.round(coef_std_large, 4))

The coefficient *mean* barely moves between 12 and 300 applicants — both are close to the true underlying relationship. What changes dramatically is the *std*: the income coefficient's uncertainty shrinks from 0.0905 down to 0.0136 as evidence accumulates. A plain LinearRegression or Ridge fit would report only the mean column at either sample size, with no way to distinguish "confidently 2.10" from "our best guess is 2.14, but with only 12 data points, anywhere from roughly 1.9 to 2.3 is plausible."

## The Gaussian Process — Bayesian Inference Over Entire Functions

A Gaussian Process (GP) extends the same idea from a handful of coefficients to an *entire regression function*. Instead of assuming a fixed functional form (linear, polynomial degree 3, ...), a GP places a prior directly over the space of possible functions, and the **kernel** controls what "plausible" functions look like — how smooth they are, how quickly they can change.

$$f(x) \sim \mathcal{GP}(m(x), k(x,x')) \qquad m = \text{mean function (often 0)} \quad k = \text{covariance/kernel function}$$

The RBF kernel k(x,x′) = σ² · exp(−‖x−x′‖² / 2ℓ²) says: points close together in x-space should have similar, correlated function values; points far apart become roughly independent. This single assumption is what lets a GP predict a smooth curve through sparse data *and* honestly report much higher uncertainty the further a query point sits from anything actually observed:

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel as C

# Flipkart: festival-season demand multiplier vs. days-to-Diwali — only 6 observed points
days_to_diwali = np.array([60,45,30,20,10,3]).reshape(-1,1).astype(float)
demand_multiplier = np.array([1.05,1.15,1.35,1.7,2.4,3.1])

kernel = C(1.0) * RBF(length_scale=15, length_scale_bounds=(1,100)) + WhiteKernel(noise_level=0.01)
gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, normalize_y=True, random_state=3)
gp.fit(days_to_diwali, demand_multiplier)

# A mix of interpolation (inside the observed 3-60 day range) and extrapolation (well outside it)
test_days = np.array([35,15,2,90,120]).reshape(-1,1).astype(float)
mean_pred, std_pred = gp.predict(test_days, return_std=True)
for d, m, s in zip(test_days.ravel(), mean_pred, std_pred):
    print(f"days={d:5.1f}  predicted={m:.3f}  std={s:.3f}")

Notice the std column: for the three days that sit inside the observed 3–60 day range, uncertainty stays tiny (0.03–0.04). The moment the query point moves to 90 or 120 days out — well past anything actually observed — std jumps to 0.82 and then 1.87. A Random Forest or XGBoost model asked to predict at day 120 would return a confident-looking single number with no built-in signal that it's extrapolating wildly; the GP's honest answer is closer to "roughly 1.7, but really, we have no idea — we've never seen data out this far."

## Practical Aspects

| Consideration | Detail |
|---|---|
| Kernel choice | RBF assumes smooth functions; Matérn kernels allow rougher functions; periodic kernels for seasonal patterns (festival cycles, weekly demand) |
| Hyperparameter fitting | Kernel hyperparameters (length-scale, noise level) are optimized by maximizing the *marginal likelihood* — automatic, no separate validation set required |
| Computational cost | Exact GP inference costs O(n³) — the reason GPs are typically used on hundreds to a few thousand points, not millions, unlike Random Forest or XGBoost |
| Where GPs are used in practice | Bayesian Optimization (tuning expensive-to-evaluate hyperparameters or physical experiments), spatial statistics/kriging, small-data scientific and engineering regression problems |

> **💡 When to Reach for Bayesian Methods**
>
> Prefer Bayesian Linear Regression / Gaussian Processes over the models used elsewhere in this course specifically when: (1) the dataset is small enough that point estimates alone would be overconfident, (2) a genuine, calibrated uncertainty estimate is itself part of the deliverable — not just the prediction — or (3) the next decision (which experiment to run next, which hyperparameter to try next) depends on knowing *where* the model is most uncertain, which is exactly what powers Bayesian Optimization.

## Try It — Drag the Kernel's Length-Scale and Watch the Band Respond

The same 6 Flipkart demand points, refit live as you move the length-scale slider. A short length-scale lets the curve wiggle and trusts each point locally (tight band everywhere, including gaps); a long length-scale forces a smoother curve and reverts faster to the prior the moment you leave the observed 3–60 day range. Dashed guides mark the five test days from the printed table above so you can compare directly.

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · More data, tighter uncertainty

Fit `BayesianRidge` on the small and the large table from the lesson. Store the **standard deviation of the first coefficient** for each in `std_small` and `std_large` (`np.sqrt(np.diag(model.sigma_))[0]`), and `more_data_helps` = `std_large < std_small`.

In [ ]:
std_small = std_large = more_data_helps = None   # TODO (reuse the arrays from the lesson)


In [ ]:
try:
    check("large-sample uncertainty is smaller", more_data_helps is True)
    check("values positive", std_small > 0 and std_large > 0)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
from sklearn.linear_model import BayesianRidge
std = lambda X, y: np.sqrt(np.diag(BayesianRidge().fit(X, y).sigma_))[0]
std_small, std_large = std(income_cibil_small, loan_amount_small), std(income_cibil_large, loan_amount_large)
more_data_helps = bool(std_large < std_small)

```

</details>

### Exercise 2 · Medium · A conjugate Beta update

Meridian Bank's prior belief about a campaign's conversion rate is Beta(2, 2). After 7 conversions in 10 contacts, write `update(a, b, successes, trials)` returning the posterior `(a, b)`. Then store the posterior mean in `post_mean`.

In [ ]:
def update(a, b, successes, trials):
    pass   # TODO
post_mean = None


In [ ]:
try:
    a, b = update(2, 2, 7, 10)
    check("posterior parameters", (a, b) == (9, 5))
    check("posterior mean 9/14", abs(post_mean - 9 / 14) < 1e-12)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
def update(a, b, successes, trials):
    return a + successes, b + (trials - successes)
a, b = update(2, 2, 7, 10)
post_mean = a / (a + b)

```

</details>

### Exercise 3 · Stretch · Credible intervals from a Gaussian process

Fit a `GaussianProcessRegressor` (constant × RBF + white noise) on the lesson's 6 Diwali-demand points. For `days = 30` (inside the data) and `days = 120` (far outside) compute the 95% interval `mean ± 1.96·std`. Store the widths in `width_in` and `width_out`; extrapolation should be **less certain**.

In [ ]:
width_in = width_out = None   # TODO (reuse `gp` from the lesson)


In [ ]:
try:
    check("extrapolating is less certain", width_out > width_in)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
m_in, s_in = gp.predict([[30.0]], return_std=True)
m_out, s_out = gp.predict([[120.0]], return_std=True)
width_in, width_out = 2 * 1.96 * s_in[0], 2 * 1.96 * s_out[0]

```

Error bars that widen away from the data are the honest answer to 'how sure are you?'.

</details>

---
*Back to the course: **Machine Learning End To End → The Bayesian Approach & Gaussian Processes**.*